# **Computing $\mathcal{R}_0$ with `pyR0compute`**

This notebook presents several examples of the symbolic computation of the ***basic reproduction number*** using the **next-generation matrix** method (van den Driessche and Watmough, 2002), in which $\mathcal{R}_0 = \rho(FV^{-1})$.

The user only needs to write the **ODE system** and specify the **infected compartments**. The library automatically recognizes as parameters all symbols that are neither state variables nor auxiliary expressions, computes the disease-free equilibrium, and separates the new-infection terms ($\mathcal{F}$) from the transition terms ($\mathcal{V}$). Since this decomposition is not unique, the automatic classification can be overridden with `new_infections={...}`, and the equilibrium, when it is not determined by the equations, can be set with `dfe={...}`.

The examples cover classical epidemiological models, vector-borne transmission, within-host dynamics, and strain competition. In addition, normalized sensitivity indices are computed and reports of the procedure are generated as $\LaTeX$ code, ready to paste into a document.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yvan-BM-Git/pyR0compute/blob/main/examples/legacy/01_examples.ipynb)

In [1]:
%pip install pyR0compute                      # Installs the pyR0compute library from PyPI

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sympy as sp                            # Imports SymPy for symbolic computation
from pyr0compute import R0Model               # Imports the R0Model class from the pyr0compute package
from IPython.display import Markdown, display # Imports Markdown and display to render results as Markdown in Jupyter Notebook

## 1. SEIR model

$$
\begin{cases}
\dfrac{dS}{dt} = \Lambda - \beta SI - \mu S \\[6pt]
\dfrac{dE}{dt} = \beta SI - (\sigma + \mu)E \\[6pt]
\dfrac{dI}{dt} = \sigma E - (\gamma + \mu)I \\[6pt]
\dfrac{dR}{dt} = \gamma I - \mu R
\end{cases}
$$

Only the equations and the infected compartments are provided.

In [3]:
seir = R0Model("""
    dS/dt = Lambda - beta*S*I - mu*S
    dE/dt = beta*S*I - (sigma + mu)*E
    dI/dt = sigma*E - (gamma + mu)*I
    dR/dt = gamma*I - mu*R
""", infected=["E", "I"])

seir.R0

Lambda*beta*sigma/(mu*(gamma + mu)*(mu + sigma))

In [4]:
print("Detected parameters:", seir.parameters)
print("Disease-free equilibrium:", seir.dfe)

Detected parameters: (Lambda, beta, gamma, mu, sigma)
Disease-free equilibrium: {S: Lambda/mu, E: 0, I: 0, R: 0}


The matrices $F$, $V$ and $K = FV^{-1}$ are available as SymPy objects:

In [5]:
seir.F, seir.V, seir.K

(Matrix([
 [0, Lambda*beta/mu],
 [0,              0]]),
 Matrix([
 [mu + sigma,          0],
 [    -sigma, gamma + mu]]),
 Matrix([
 [Lambda*beta*sigma/(mu*(gamma + mu)*(mu + sigma)), Lambda*beta/(mu*(gamma + mu))],
 [                                               0,                             0]]))

In [6]:
display(Markdown(seir.report_latex("markdown")))

## pyR0compute: next-generation matrix method

**Variables:** $S, E, I, R$  
**Infected compartments:** $E, I$  
**Parameters (detected automatically):** $\Lambda, \beta, \gamma, \mu, \sigma$

### Model

$$
\begin{aligned}
\frac{dS}{dt} &= - I S \beta + \Lambda - S \mu \\
\frac{dE}{dt} &= - E \left(\mu + \sigma\right) + I S \beta \\
\frac{dI}{dt} &= E \sigma - I \left(\gamma + \mu\right) \\
\frac{dR}{dt} &= I \gamma - R \mu
\end{aligned}
$$

### New-infection terms (automatic)

$$
\begin{aligned}
\mathcal{F}_{E} &= I S \beta \\
\mathcal{F}_{I} &= 0
\end{aligned}
$$

### Term classification

| Equation | Term | Class | Reason |
|---|---|:---:|---|
| $dE/dt$ | $E \left(- \mu - \sigma\right)$ | $\mathcal{V}$ | negative term |
| $dE/dt$ | $I S \beta$ | $\mathcal{F}$ | transfer from uninfected S |
| $dI/dt$ | $E \sigma$ | $\mathcal{V}$ | transfer from infected E |
| $dI/dt$ | $I \left(- \gamma - \mu\right)$ | $\mathcal{V}$ | negative term |

### Transition terms

$$
\begin{aligned}
\mathcal{V}_{E} &= E \mu + E \sigma \\
\mathcal{V}_{I} &= - E \sigma + I \gamma + I \mu
\end{aligned}
$$

### Disease-free equilibrium

$$
\begin{aligned}
S &= \frac{\Lambda}{\mu} \\
E &= 0 \\
I &= 0 \\
R &= 0
\end{aligned}
$$

### Next-generation matrix

$$
\begin{aligned}
F &= \begin{bmatrix}0 & \frac{\Lambda \beta}{\mu}\\0 & 0\end{bmatrix} \\
V &= \begin{bmatrix}\mu + \sigma & 0\\- \sigma & \gamma + \mu\end{bmatrix} \\
K = F V^{-1} &= \begin{bmatrix}\frac{\Lambda \beta \sigma}{\mu \left(\gamma + \mu\right) \left(\mu + \sigma\right)} & \frac{\Lambda \beta}{\mu \left(\gamma + \mu\right)}\\0 & 0\end{bmatrix}
\end{aligned}
$$

Only $E$ receive new infections, so $\mathcal{R}_0$ is the spectral radius of the next-generation matrix with small domain:

$$
\begin{aligned}
K_{\text{small}} &= \begin{bmatrix}\frac{\Lambda \beta \sigma}{\mu \left(\gamma + \mu\right) \left(\mu + \sigma\right)}\end{bmatrix}
\end{aligned}
$$

### Basic reproduction number

$$
\begin{aligned}
\mathcal{R}_0 = \rho\left(F V^{-1}\right) &= \frac{\Lambda \beta \sigma}{\mu \left(\gamma + \mu\right) \left(\mu + \sigma\right)}
\end{aligned}
$$

In [7]:
print(seir.report_latex())

\section*{pyR0compute: next-generation matrix method}

\textbf{Variables:} $S, E, I, R$ \\
\textbf{Infected compartments:} $E, I$ \\
\textbf{Parameters (detected automatically):} $\Lambda, \beta, \gamma, \mu, \sigma$

\subsection*{Model}

\begin{align*}
\frac{dS}{dt} &= - I S \beta + \Lambda - S \mu \\
\frac{dE}{dt} &= - E \left(\mu + \sigma\right) + I S \beta \\
\frac{dI}{dt} &= E \sigma - I \left(\gamma + \mu\right) \\
\frac{dR}{dt} &= I \gamma - R \mu
\end{align*}

\subsection*{New-infection terms (automatic)}

\begin{align*}
\mathcal{F}_{E} &= I S \beta \\
\mathcal{F}_{I} &= 0
\end{align*}

\subsection*{Term classification}

\begin{tabular}{llcl}
\hline
Equation & Term & Class & Reason \\
\hline
$dE/dt$ & $E \left(- \mu - \sigma\right)$ & $\mathcal{V}$ & negative term \\
$dE/dt$ & $I S \beta$ & $\mathcal{F}$ & transfer from uninfected S \\
$dI/dt$ & $E \sigma$ & $\mathcal{V}$ & transfer from infected E \\
$dI/dt$ & $I \left(- \gamma - \mu\right)$ & $\mathcal{V}$ & negative term \\


In [8]:
print(seir.latex())

\frac{\Lambda \beta \sigma}{\mu \left(\gamma + \mu\right) \left(\mu + \sigma\right)}


### Sensitivity indices

$\Upsilon_p = \dfrac{\partial R_0}{\partial p}\dfrac{p}{R_0}$: a 1% increase in $p$ changes $R_0$ by $\Upsilon_p$%.

In [9]:
seir.sensitivity_indices()

{Lambda: 1,
 beta: 1,
 gamma: -gamma/(gamma + mu),
 mu: -(2*gamma*mu + gamma*sigma + 3*mu**2 + 2*mu*sigma)/((gamma + mu)*(mu + sigma)),
 sigma: mu/(mu + sigma)}

In [10]:
values = {"Lambda": 10, "beta": 0.001, "sigma": 0.2, "gamma": 0.1, "mu": 0.02}
print("R0 =", seir.R0_numeric(values))
seir.sensitivity_indices(values)

R0 = 3.787878787878788


{Lambda: 1.0,
 beta: 1.0,
 gamma: -0.8333333333333333,
 mu: -1.2575757575757578,
 sigma: 0.09090909090909093}

## **2. Classical SIR model (closed population)**

$$
\begin{cases}
\dfrac{dS}{dt}
= -\dfrac{\beta S I}{N} \\[8pt]

\dfrac{dI}{dt}
= \dfrac{\beta S I}{N} - \gamma I \\[8pt]

\dfrac{dR}{dt}
= \gamma I
\end{cases}
$$

### Model conditions

- **Infected compartment:** $I$.
- **Disease-free equilibrium (DFE):**
  $$S = N,\qquad I = 0,\qquad R = 0.$$

In [11]:
sir = R0Model({"S": "-beta*S*I/N",
               "I": "beta*S*I/N - gamma*I",
               "R": "gamma*I"}, infected=["I"], dfe={"S": "N"})
sir.R0

beta/gamma

Without `dfe`, the error message states exactly what is missing:

In [12]:
try:
    R0Model({"S": "-beta*S*I/N", 
             "I": "beta*S*I/N - gamma*I", 
             "R": "gamma*I"}, infected=["I"])
except Exception as e:
    print(type(e).__name__, "->", e)

DiseaseFreeEquilibriumError -> The disease-free value of ['S'] is not determined by the equations (e.g. a closed population without births). Provide it with dfe={'S': ...}.


## **3. Vector-borne disease: Ross-Macdonald model**

$$
\begin{cases}
\dfrac{dS_h}{dt}
= L_h - \dfrac{a b_h S_h I_v}{N_h} - \mu_h S_h \\[8pt]

\dfrac{dI_h}{dt}
= \dfrac{a b_h S_h I_v}{N_h}
- (\gamma + \mu_h) I_h \\[8pt]

\dfrac{dR_h}{dt}
= \gamma I_h - \mu_h R_h \\[8pt]

\dfrac{dS_v}{dt}
= L_v - \dfrac{a b_v S_v I_h}{N_h} - \mu_v S_v \\[8pt]

\dfrac{dI_v}{dt}
= \dfrac{a b_v S_v I_h}{N_h} - \mu_v I_v
\end{cases}
$$

### State variables

- $S_h$: number of **susceptible hosts**.
- $I_h$: number of **infected hosts**.
- $R_h$: number of **recovered hosts**.
- $S_v$: number of **susceptible vectors**.
- $I_v$: number of **infected vectors**.
- $N_h = S_h + I_h + R_h$: total host population.


In [13]:
rm = R0Model("""
    dS_h/dt = L_h - a*b_h*S_h*I_v/N_h - mu_h*S_h
    dI_h/dt = a*b_h*S_h*I_v/N_h - (gamma + mu_h)*I_h
    dR_h/dt = gamma*I_h - mu_h*R_h
    dS_v/dt = L_v - a*b_v*S_v*I_h/N_h - mu_v*S_v
    dI_v/dt = a*b_v*S_v*I_h/N_h - mu_v*I_v
""", infected=["I_h", "I_v"])
rm.new_infections

{I_h: I_v*S_h*a*b_h/N_h, I_v: I_h*S_v*a*b_v/N_h}

In [14]:
rm.K

Matrix([
[                                  0, L_h*a*b_h/(N_h*mu_h*mu_v)],
[L_v*a*b_v/(N_h*mu_v*(gamma + mu_h)),                         0]])

In [15]:
rm.R0

sqrt(L_h)*sqrt(L_v)*a*sqrt(b_h)*sqrt(b_v)/(N_h*sqrt(mu_h)*mu_v*sqrt(gamma + mu_h))

## **4. Host-vector model with human-to-human transmission and logistic vector growth**

$$
\begin{cases}
N_h = S_h + E_h + I_h + R_h \\[6pt]

\dfrac{dS_h}{dt}
= \Delta_h
- \dfrac{\delta \alpha_{hv} S_h I_v}{N_h}
- \dfrac{\beta \alpha_{hh} S_h I_h}{N_h}
- (\theta + \mu_h)S_h \\[8pt]

\dfrac{dE_h}{dt}
= \dfrac{\delta \alpha_{hv} S_h I_v}{N_h}
+ \dfrac{\beta \alpha_{hh} S_h I_h}{N_h}
- (\sigma + \mu_h)E_h \\[8pt]

\dfrac{dI_h}{dt}
= \sigma E_h
- (\gamma + \mu_h)I_h \\[8pt]

\dfrac{dR_h}{dt}
= \gamma I_h
- \mu_h R_h
+ \theta S_h \\[8pt]

\dfrac{dM_v}{dt}
= \phi_v M_v
\left(1-\dfrac{M_v}{\kappa_v}\right)
- (\omega+\mu_m)M_v \\[8pt]

\dfrac{dS_v}{dt}
= \omega M_v
- \dfrac{\delta\alpha_{vh}S_vI_h}{N_h}
- (\mu_v+\zeta)S_v \\[8pt]

\dfrac{dE_v}{dt}
= \dfrac{\delta\alpha_{vh}S_vI_h}{N_h}
- (\psi+\mu_v+\zeta)E_v \\[8pt]

\dfrac{dI_v}{dt}
= \psi E_v
- (\mu_v+\zeta)I_v
\end{cases}
$$

### State variables

#### Human hosts

- $S_h$: number of **susceptible human hosts**.
- $E_h$: number of **exposed human hosts**, infected but not yet infectious.
- $I_h$: number of **infected and infectious human hosts**.
- $R_h$: number of **recovered human hosts**.
- $N_h = S_h + E_h + I_h + R_h$: **total human population**.

#### Vectors

- $M_v$: population of **immature vectors**.
- $S_v$: number of **susceptible adult vectors**.
- $E_v$: number of **exposed adult vectors**, infected but not yet infectious.
- $I_v$: number of **infected and infectious adult vectors**.

The total human population is defined as an auxiliary expression. The model has two disease-free equilibria (with and without vectors); the one with vectors is selected, and both remain available in `hv.dfe_candidates`.

In [16]:
hv = R0Model("""
    N_h = S_h + E_h + I_h + R_h
    dS_h/dt = Delta_h - delta*alpha_hv*S_h*I_v/N_h - beta*alpha_hh*S_h*I_h/N_h - (theta + mu_h)*S_h
    dE_h/dt = delta*alpha_hv*S_h*I_v/N_h + beta*alpha_hh*S_h*I_h/N_h - (sigma + mu_h)*E_h
    dI_h/dt = sigma*E_h - (gamma + mu_h)*I_h
    dR_h/dt = gamma*I_h - mu_h*R_h + theta*S_h
    dM_v/dt = phi_v*M_v*(1 - M_v/kappa_v) - (omega + mu_m)*M_v
    dS_v/dt = omega*M_v - delta*alpha_vh*S_v*I_h/N_h - (mu_v + zeta)*S_v
    dE_v/dt = delta*alpha_vh*S_v*I_h/N_h - (psi + mu_v + zeta)*E_v
    dI_v/dt = psi*E_v - (mu_v + zeta)*I_v
""", infected=["E_h", "I_h", "E_v", "I_v"])
hv.dfe

/tmp/ipykernel_1240/135047437.py:1: UserWarning: 2 disease-free equilibria were found; using the one with the most non-zero compartments: {'S_h': Delta_h/(mu_h + theta), 'R_h': Delta_h*theta/(mu_h*(mu_h + theta)), 'M_v': -kappa_v*(mu_m + omega - phi_v)/phi_v, 'S_v': -kappa_v*omega*(mu_m + omega - phi_v)/(phi_v*(mu_v + zeta))}. Pass dfe={...} to choose another one (all are in model.dfe_candidates).
  hv = R0Model("""
/tmp/ipykernel_1240/135047437.py:1: UserWarning: Entry F[2,1] = -alpha_vh*delta*kappa_v*mu_h*omega*(mu_m + omega - phi_v)/(Delta_h*phi_v*(mu_v + zeta)) is negative; new-infection terms should be non-negative.
  hv = R0Model("""


{S_h: Delta_h/(mu_h + theta),
 E_h: 0,
 I_h: 0,
 R_h: Delta_h*theta/(mu_h*(mu_h + theta)),
 M_v: -kappa_v*(mu_m + omega - phi_v)/phi_v,
 S_v: -kappa_v*omega*(mu_m + omega - phi_v)/(phi_v*(mu_v + zeta)),
 E_v: 0,
 I_v: 0}

Only $E_h$ and $E_v$ receive new infections, so $R_0$ is the spectral radius of the next-generation matrix with small domain, of size $2\times 2$:

In [17]:
hv.next_generation_matrix_small

Matrix([
[                                                    alpha_hh*beta*mu_h*sigma/((gamma + mu_h)*(mu_h + sigma)*(mu_h + theta)), alpha_hv*delta*mu_h*psi/((mu_h + theta)*(mu_v + zeta)*(mu_v + psi + zeta))],
[-alpha_vh*delta*kappa_v*mu_h*omega*sigma*(mu_m + omega - phi_v)/(Delta_h*phi_v*(gamma + mu_h)*(mu_h + sigma)*(mu_v + zeta)),                                                                          0]])

In [18]:
hv.R0

alpha_hh*beta*mu_h*sigma/(2*(gamma + mu_h)*(mu_h + sigma)*(mu_h + theta)) + sqrt(alpha_hh**2*beta**2*mu_h**2*sigma**2/((gamma + mu_h)**2*(mu_h + sigma)**2*(mu_h + theta)**2) - 4*alpha_hv*alpha_vh*delta**2*kappa_v*mu_h**2*omega*psi*sigma*(mu_m + omega - phi_v)/(Delta_h*phi_v*(gamma + mu_h)*(mu_h + sigma)*(mu_h + theta)*(mu_v + zeta)**2*(mu_v + psi + zeta)))/2

## **5. Within-host model: target cells, infected cells and virus**

The model describes the dynamics of a viral infection within the host, considering three populations: target cells susceptible to infection, infected cells, and free viral particles.

$$
\begin{cases}
\dfrac{dT}{dt}
= s-dT-\beta TV \\[8pt]

\dfrac{dI}{dt}
= \beta TV-\delta I \\[8pt]

\dfrac{dV}{dt}
= pI-cV
\end{cases}
$$

### State variables

* $T$: number or concentration of **target cells susceptible** to infection.
* $I$: number or concentration of **infected cells**.
* $V$: number or concentration of **free viral particles**.


In [19]:
tiv = R0Model("""
    dT/dt = s - d*T - beta*T*V
    dI/dt = beta*T*V - delta*I
    dV/dt = p*I - c*V
""", infected=["I", "V"])
tiv.R0

beta*p*s/(c*d*delta)

## **6. van den Driessche and Watmough (2002), model with treatment (§4.1)**

$$
\begin{cases}
N = S + E + I + T \\[6pt]

\dfrac{dE}{dt}
= \dfrac{\beta_1 S I}{N}
+ \dfrac{\beta_2 T I}{N}
- (d+\nu+r_1)E
+ p r_2 I \\[8pt]

\dfrac{dI}{dt}
= \nu E
- (d+r_2)I \\[8pt]

\dfrac{dS}{dt}
= b-dS
- \dfrac{\beta_1 S I}{N} \\[8pt]

\dfrac{dT}{dt}
= -dT
+ r_1E
+ q r_2I
- \dfrac{\beta_2 T I}{N}
\end{cases}
$$

### State variables

- $S$: number of **susceptible individuals**.
- $E$: number of **exposed individuals**, infected but not yet infectious.
- $I$: number of **infected and infectious individuals**.
- $T$: number of **individuals under treatment**.
- $N = S+E+I+T$: **total population**.

In [20]:
tb = R0Model("""
    N = S + E + I + T
    dE/dt = beta1*S*I/N + beta2*T*I/N - (d + nu + r1)*E + p*r2*I
    dI/dt = nu*E - (d + r2)*I
    dS/dt = b - d*S - beta1*S*I/N
    dT/dt = -d*T + r1*E + q*r2*I - beta2*T*I/N
""", infected=["E", "I"])
tb.R0

-beta1*nu/(-d**2 - d*nu - d*r1 - d*r2 + nu*p*r2 - nu*r2 - r1*r2)

## **7. Two strains with superinfection**

The model considers a susceptible population and two infectious strains that compete with each other. The interaction between strains is represented by a **superinfection** term $\nu I_1 I_2$. The dominant strain depends on the epidemiological parameters, so the basic reproduction number of the system is given by the larger of the two strain-specific reproduction numbers.

$$
\begin{cases}
\dfrac{dI_1}{dt}
= \beta_1 I_1 S
- (b+\gamma_1)I_1
+ \nu I_1 I_2 \\[8pt]

\dfrac{dI_2}{dt}
= \beta_2 I_2 S
- (b+\gamma_2)I_2
- \nu I_1 I_2 \\[8pt]

\dfrac{dS}{dt}
= b-bS
+\gamma_1 I_1
+\gamma_2 I_2
-(\beta_1I_1+\beta_2I_2)S
\end{cases}
$$

### State variables

- $S$: proportion or number of **susceptible individuals**.
- $I_1$: proportion or number of individuals **infected with strain 1**.
- $I_2$: proportion or number of individuals **infected with strain 2**.

The strain with the larger reproduction number has an epidemiological advantage and, under the model assumptions, determines which of the two strains can become established in the population.

In [21]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    two = R0Model("""
        dI1/dt = beta1*I1*S - (b + gamma1)*I1 + nu*I1*I2
        dI2/dt = beta2*I2*S - (b + gamma2)*I2 - nu*I1*I2
        dS/dt = b - b*S + gamma1*I1 + gamma2*I2 - (beta1*I1 + beta2*I2)*S
    """, infected=["I1", "I2"])
    R0 = two.R0
R0

Max(beta1/(b + gamma1), beta2/(b + gamma2))

## **8. CHALLENGE: Compute $R_0$ for your own model**

Now apply the method to an **epidemiological model of your choice**. Replace the equations in the example with those of your model and specify the compartments you consider infected.

If `R0Model` does not correctly identify the **new-infection** and **transition** terms, you can define them explicitly with `new_infections={...}`. Likewise, if you need to set the **disease-free equilibrium (DFE)** manually, you can use `dfe={...}`.

In [22]:
# Example to start with: replace these equations with those of your model
# and "infected" with the compartments you consider infected.
model = R0Model("""
    dS/dt = Lambda - beta*S*I - mu*S
    dI/dt = beta*S*I - (gamma + mu)*I
    dR/dt = gamma*I - mu*R
""", infected=["I"])
model.R0

Lambda*beta/(mu*(gamma + mu))

In [23]:
print(model.latex())

\frac{\Lambda \beta}{\mu \left(\gamma + \mu\right)}
